# Import dati:

In [3]:
import json
import pandas as pd
import numpy as np
from random import randrange
from matplotlib import pyplot as plt
import seaborn as sns

In [4]:
#C:/Users/Francesco/Documents/uni/tesi/data_expl_analysis
#C:/Users/fra24/OneDrive/Documenti/uni/magistrale/tesi/data/problems_2016.json
df = pd.read_csv('C:/Users/Francesco/Documents/uni/tesi/data_expl_analysis/moonboard_2016_cleaned_ready_for_masking.csv', encoding='utf-8')
df.iloc[:,0:9].head(10)
df.shape

(59506, 215)

# Feature Engeneering:

In questa sezione andremo a creare delle maschere sulla matrice per poter aggiungere informazione all nostro dataset

In [5]:
holds = df.loc[:, "A1":"K18"]
holds = (holds != 0).astype(int)
holds

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,K9,K10,K11,K12,K13,K14,K15,K16,K17,K18
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59501,0,0,0,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
59502,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59503,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59504,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Feature Engeneering:

In [8]:
# Feature engineering completo per matrice 18x11 binaria (arrampicata)
import numpy as np
from scipy.ndimage import center_of_mass, label, uniform_filter, gaussian_filter, laplace
from scipy.stats import skew, entropy
from scipy.stats import skew
from skimage.util import view_as_windows

In [9]:
def safe_skew(arr):
        if np.std(arr) < 1e-8:
            return 0.0  # oppure np.nan
        return skew(arr)

def estrai_feature_complete_con_nomi(matrice):
    features = []
    feature_names = []

    # --- Globali base ---
    totale = np.sum(matrice)
    centroide = center_of_mass(matrice)
    coords = np.argwhere(matrice)
    var_y, var_x = (np.var(coords[:, 0]), np.var(coords[:, 1])) if coords.size > 0 else (0, 0)
    sim_vert = np.mean(matrice[:, :5] == np.fliplr(matrice[:, 6:]))
    sim_oriz = np.mean(matrice[:9, :] == np.flipud(matrice[9:, :]))
    diag_1 = np.trace(matrice)
    diag_2 = np.trace(np.fliplr(matrice))
    media_loc_u_3 = uniform_filter(matrice.astype(float), size=3)
    media_loc_u_5 = uniform_filter(matrice.astype(float), size=5)
    media_loc_g = gaussian_filter(matrice.astype(float), sigma=1)
    media_loc_l = laplace(matrice.astype(float))

    features += [totale, *centroide, var_y, var_x, sim_vert, sim_oriz, diag_1, diag_2]
    feature_names += [
        "totale", "centroide_y", "centroide_x", "var_y", "var_x",
        "sim_vert", "sim_oriz", "diag_1", "diag_2"
    ]
    features += [np.mean(media_loc_u_3), np.std(media_loc_u_3), np.mean(media_loc_u_5), np.std(media_loc_u_5),
                 np.mean(media_loc_g), np.std(media_loc_g), np.mean(media_loc_l), np.std(media_loc_l)]
    feature_names += ["media_loc_u_3_mean", "media_loc_u_3_std", "media_loc_u_5_mean", "media_loc_u_5_std",
                      "media_loc_g_mean", "media_loc_g_std", "media_loc_l_mean", "media_loc_l_std"]
    features += [np.count_nonzero(np.sum(matrice, axis=0)), np.count_nonzero(np.sum(matrice, axis=1))]
    feature_names += ["colonne_usate", "righe_usate"]

    # --- Segmentazioni orizzontali e verticali ---

    # Segmentazioni sulle righe: 2, 3, 4, 6, 9, 18 bande
    for n in [2, 3, 4, 6, 9, 18]:
        for i in range(n):
            start = i * 18 // n
            end = (i + 1) * 18 // n if i < n - 1 else 18  # l'ultimo gruppo arriva fino in fondo
            features += [np.mean(matrice[start:end, :])]
            feature_names += [f"mean_rowband_{n}_{i+1}"]

# Segmentazioni sulle colonne: 2, 3, 4, 5, 11 bande
    for n in [2, 3, 4, 5, 11]:
        for i in range(n):
            start = i * 11 // n
            end = (i + 1) * 11 // n if i < n - 1 else 11  # l'ultimo gruppo arriva fino in fondo
            features += [np.mean(matrice[:, start:end])]
            feature_names += [f"mean_colband_{n}_{i+1}"]

    # --- Blocchi fissi (macro-zone 6x4, 9x4, 6x2, 3x3) ---

    # Blocchi 6x4
    for i in range(0, 18, 6):
        for j in range(0, 11, 4):
            blocco = matrice[i:i+6, j:j+4]
            if blocco.shape == (6,4):
                blocco_coords = np.argwhere(blocco)
                #blocco_centroide = center_of_mass(blocco) if np.any(blocco) else (0, 0)
                blocco_var = np.var(blocco_coords[:, 0]) + np.var(blocco_coords[:, 1]) if blocco_coords.size else 0
                features += [np.mean(blocco), blocco_var]
                feature_names += [
                    f"blocco6x4_{i}_{j}_mean",
                    f"blocco6x4_{i}_{j}_var"
                ]
    # Blocchi 9x4
    for i in range(0, 18, 9):
        for j in range(0, 11, 4):
            blocco = matrice[i:i+9, j:j+4]
            if blocco.shape == (9,4):
                blocco_coords = np.argwhere(blocco)
                #blocco_centroide = center_of_mass(blocco) if np.any(blocco) else (0, 0)
                blocco_var = np.var(blocco_coords[:, 0]) + np.var(blocco_coords[:, 1]) if blocco_coords.size else 0
                features += [np.mean(blocco), blocco_var]
                feature_names += [
                    f"blocco9x4_{i}_{j}_mean",
                    f"blocco9x4_{i}_{j}_var"
                ]
    # Blocchi 6x2
    for i in range(0, 18, 6):
        for j in range(0, 11, 2):
            blocco = matrice[i:i+6, j:j+2]
            if blocco.shape == (6,2):
                blocco_coords = np.argwhere(blocco)
                #blocco_centroide = center_of_mass(blocco) if np.any(blocco) else (0, 0)
                blocco_var = np.var(blocco_coords[:, 0]) + np.var(blocco_coords[:, 1]) if blocco_coords.size else 0
                features += [np.mean(blocco), blocco_var]
                feature_names += [
                    f"blocco6x2_{i}_{j}_mean",
                    f"blocco6x2_{i}_{j}_var"
                ]
    # Blocchi 3x3
    for i in range(0, 18, 3):
        for j in range(0, 11, 3):
            blocco = matrice[i:i+3, j:j+3]
            if blocco.shape == (3,3):
                blocco_coords = np.argwhere(blocco)
                #blocco_centroide = center_of_mass(blocco) if np.any(blocco) else (0, 0)
                blocco_var = np.var(blocco_coords[:, 0]) + np.var(blocco_coords[:, 1]) if blocco_coords.size else 0
                features += [np.mean(blocco), blocco_var]
                feature_names += [
                    f"blocco3x3_{i}_{j}_mean",
                    f"blocco3x3_{i}_{j}_var"
                ]

    # Blocchi 6x3 (verticali, copertura completa)
    for i in [0, 6, 12]:
        i_end = 18 if i == 12 else i + 6
        for j in [0, 3, 6, 9]:
            j_end = 11 if j == 9 else j + 3
            blocco = matrice[i:i_end, j:j_end]
            if blocco.shape == (i_end - i, j_end - j):
                blocco_coords = np.argwhere(blocco)
                #blocco_centroide = center_of_mass(blocco) if np.any(blocco) else (0, 0)
                blocco_var = np.var(blocco_coords[:, 0]) + np.var(blocco_coords[:, 1]) if blocco_coords.size else 0
                features += [np.mean(blocco), blocco_var]
                feature_names += [
                    f"blocco6x3_{i}_{j}_mean",
                    f"blocco6x3_{i}_{j}_var"
                ]
    
    # Blocchi 9x2 (orizzontali, copertura completa)
    for i in [0, 9]:
        i_end = 18 if i == 9 else i + 9
        for j in [0, 2, 4, 6, 8, 10]:
            j_end = 11 if j == 10 else j + 2
            blocco = matrice[i:i_end, j:j_end]
            if blocco.shape == (i_end - i, j_end - j):
                blocco_coords = np.argwhere(blocco)
                #blocco_centroide = center_of_mass(blocco) if np.any(blocco) else (0, 0)
                blocco_var = np.var(blocco_coords[:, 0]) + np.var(blocco_coords[:, 1]) if blocco_coords.size else 0
                features += [np.mean(blocco), blocco_var]
                feature_names += [
                    f"blocco9x2_{i}_{j}_mean",
                    f"blocco9x2_{i}_{j}_var"
                ]

    # Skewness e entropia
    skew_y = safe_skew(np.sum(matrice, axis=1))
    skew_x = safe_skew(np.sum(matrice, axis=0))
    p_col = np.sum(matrice, axis=0)
    p = p_col / (np.sum(p_col) + 1e-6)
    entr = entropy(p)
    features += [skew_y, skew_x, entr]
    feature_names += ["skew_y", "skew_x", "entropy"]

    # Clustering
    clusters, n_clusters = label(matrice)
    cluster_sizes = np.bincount(clusters.flatten())[1:]
    mean_cluster_size = cluster_sizes.mean() if len(cluster_sizes) > 0 else 0
    max_cluster_size = cluster_sizes.max() if len(cluster_sizes) > 0 else 0
    features += [n_clusters, mean_cluster_size, max_cluster_size, n_clusters / (totale + 1e-6)]
    feature_names += ["n_clusters", "mean_cluster_size", "max_cluster_size", "clusters_per_hold"]

    # Sliding window densità massima
    if matrice.shape[0] >= 3 and matrice.shape[1] >= 3:
        finestra = view_as_windows(matrice, (3, 3))
        dens_blocchi = np.sum(finestra, axis=(2,3))
        max_dens = np.max(dens_blocchi)
        features.append(max_dens)
        feature_names.append("max_density_3x3")
    else:
        features.append(0)
        feature_names.append("max_density_3x3")

    # Bounding box e forma
    if coords.size > 0:
        min_y, min_x = coords.min(axis=0)
        max_y, max_x = coords.max(axis=0)
        h = max_y - min_y + 1
        w = max_x - min_x + 1
        aspect_ratio = w / h if h > 0 else 0
        features += [h, w, aspect_ratio]
        feature_names += ["bbox_h", "bbox_w", "aspect_ratio"]
    else:
        features += [0, 0, 0]
        feature_names += ["bbox_h", "bbox_w", "aspect_ratio"]

    # Transizioni (switch)
    switch_righe = np.sum(np.abs(np.diff(matrice, axis=1)))
    switch_colonne = np.sum(np.abs(np.diff(matrice, axis=0)))
    features += [switch_righe, switch_colonne]
    feature_names += ["switch_righe", "switch_colonne"]

    return np.array(features), feature_names

In [10]:
# Esempio di uso su dataset
dataset = np.random.randint(0, 2, size=(100, 18, 11))
X, feature_names = zip(*[estrai_feature_complete_con_nomi(m) for m in dataset])
X = np.array(X)
df_features = pd.DataFrame(X, columns=feature_names[0])
df_features.head()

,totale,centroide_y,centroide_x,var_y,var_x,sim_vert,sim_oriz,diag_1,diag_2,media_loc_u_3_mean,...,n_clusters,mean_cluster_size,max_cluster_size,clusters_per_hold,max_density_3x3,bbox_h,bbox_w,aspect_ratio,switch_righe,switch_colonne
0,104.0,8.471154,5.125000,28.306860,9.840144,0.422222,0.555556,6.0,8.0,0.525253,...,16.0,6.500000,36.0,0.153846,7.0,18.0,11.0,0.611111,102.0,90.0
1,99.0,8.454545,4.989899,28.025712,9.949393,0.422222,0.505051,6.0,7.0,0.500000,...,15.0,6.600000,35.0,0.151515,8.0,18.0,11.0,0.611111,89.0,82.0
2,98.0,7.877551,4.673469,29.495210,11.077051,0.566667,0.494949,7.0,7.0,0.494949,...,22.0,4.454545,24.0,0.224490,9.0,18.0,11.0,0.611111,86.0,91.0
3,97.0,9.000000,4.546392,26.494845,9.814858,0.444444,0.565657,5.0,8.0,0.489899,...,18.0,5.388889,57.0,0.185567,8.0,18.0,11.0,0.611111,91.0,95.0
4,103.0,7.495146,5.019417,26.055802,9.261759,0.522222,0.525253,5.0,8.0,0.520202,...,16.0,6.437500,66.0,0.155340,8.0,18.0,11.0,0.611111,95.0,91.0


### Descrizione delle feature estratte per ogni tracciato Moonboard

Ogni osservazione rappresenta un tracciato di arrampicata su Moonboard 2016 (matrice 18x11).  
I valori 1 indicano le prese utilizzate nel tracciato.

| Nome Feature                  | Descrizione                                                                                   |
|-------------------------------|----------------------------------------------------------------------------------------------|
| **totale**                    | Numero totale di prese usate nel tracciato                                                   |
| **densita**                   | Densità di prese usate (totale / 198)                                                        |
| **centroide_y**               | Coordinata y del baricentro delle prese                                                      |
| **centroide_x**               | Coordinata x del baricentro delle prese                                                      |
| **var_y**                     | Varianza delle posizioni y delle prese                                                       |
| **var_x**                     | Varianza delle posizioni x delle prese                                                       |
| **sim_vert**                  | Simmetria verticale (tra la parte sinistra e destra della board)                             |
| **sim_oriz**                  | Simmetria orizzontale (tra la parte alta e bassa della board)                               |
| **diag_1**                    | Numero di prese sulla diagonale principale                                                   |
| **diag_2**                    | Numero di prese sulla diagonale secondaria (antidiagonale)                                  |
| **media_loc_mean**            | Media del filtro locale (media su finestre 3x3)                                              |
| **media_loc_std**             | Deviazione standard del filtro locale                                                        |
| **colonne_usate**             | Numero di colonne in cui è presente almeno una presa                                         |
| **righe_usate**               | Numero di righe in cui è presente almeno una presa                                           |

### Segmentazioni in bande

Per ogni suddivisione in bande orizzontali (righe) e verticali (colonne), vengono calcolate:

- **mean_rowband_N_I**: Media dei valori (prese) nella banda I-esima su N totali in orizzontale
- **mean_colband_N_I**: Media dei valori (prese) nella banda I-esima su N totali in verticale

Esempio:  
- `mean_colband_4_1` = media prese nella prima banda su 4 verticali

### Macro-zone (blocchi di varie dimensioni)
Per ogni blocco fisso (ad esempio 6x4, 9x4, 6x2, 3x3) vengono calcolate:

+ **bloccoXxY_I_J_mean**: Media delle prese nel blocco
+ **bloccoXxY_I_J_centroide_y**: Baricentro y delle prese nel blocco
+ **bloccoXxY_I_J_centroide_x**: Baricentro x delle prese nel blocco
+ **bloccoXxY_I_J_var**: Varianza delle posizioni delle prese nel blocco

Queste feature permettono di analizzare la distribuzione locale delle prese in diverse zone della Moonboard, a seconda della dimensione e posizione del blocco.

### Statistiche avanzate

| Nome Feature                  | Descrizione                                                                                   |
|-------------------------------|----------------------------------------------------------------------------------------------|
| **skew_y**                    | Skewness (asimmetria) della distribuzione delle prese sulle righe                            |
| **skew_x**                    | Skewness (asimmetria) della distribuzione delle prese sulle colonne                         |
| **entropy**                   | Entropia della distribuzione delle prese sulle colonne                                       |
| **n_clusters**                | Numero di cluster di prese connesse                                                          |
| **mean_cluster_size**         | Dimensione media dei cluster                                                                 |
| **max_cluster_size**          | Dimensione massima di un cluster                                                             |
| **clusters_per_hold**         | Rapporto tra numero di cluster e numero totale di prese                                      |
| **max_density_3x3**           | Massima densità di prese in una finestra 3x3                                                 |
| **bbox_h**                    | Altezza della bounding box che contiene tutte le prese                                       |
| **bbox_w**                    | Larghezza della bounding box che contiene tutte le prese                                     |
| **aspect_ratio**              | Rapporto larghezza/altezza della bounding box                                                |
| **switch_righe**              | Numero di transizioni (on/off) tra prese su tutte le righe                                   |
| **switch_colonne**            | Numero di transizioni (on/off) tra prese su tutte le colonne                                 |

**Nota:**  
Le feature `mean_rowband_*`, `sum_colband_*`, `mean_colband_*`, `blocco_*` sono generate dinamicamente per ogni suddivisione/banda/blocco, quindi il numero totale di colonne dipende da quante bande/blocchi scegli di calcolare.

### Nuove suddivisioni a blocchi: copertura completa e blocchi intelligenti
 
Per garantire che tutte le celle della Moonboard 18x11 siano incluse in almeno un blocco, la suddivisione in macro-zone ora utilizza una logica di copertura completa:
- L'ultimo blocco di ogni riga/colonna arriva sempre fino al bordo della matrice, anche se la dimensione non è perfettamente divisibile.
- Sono state aggiunte due nuove suddivisioni: **blocchi 6x3** (verticali) e **blocchi 9x2** (orizzontali), che coprono meglio la geometria della Moonboard.

**Esempio di codice per la generazione dei blocchi 6x4 (copertura completa):**
```python
for i in [0, 6, 12]:
    i_end = 18 if i == 12 else i + 6
    for j in [0, 4, 8]:
        j_end = 11 if j == 8 else j + 4
        blocco = matrice[i:i_end, j:j_end]
        # Calcolo delle feature sul blocco...
```

**Esempio per i blocchi 6x3 e 9x2:**
```python
# Blocchi 6x3 (verticali)
for i in [0, 6, 12]:
    i_end = 18 if i == 12 else i + 6
    for j in [0, 3, 6, 9]:
        j_end = 11 if j == 9 else j + 3
        blocco = matrice[i:i_end, j:j_end]
        # Calcolo delle feature sul blocco...

# Blocchi 9x2 (orizzontali)
for i in [0, 9]:
    i_end = 18 if i == 9 else i + 9
    for j in [0, 2, 4, 6, 8, 10]:
        j_end = 11 if j == 10 else j + 2
        blocco = matrice[i:i_end, j:j_end]
        # Calcolo delle feature sul blocco...
```

Questa logica assicura che nessuna riga o colonna venga esclusa e che le feature locali siano calcolate su zone omogenee e confrontabili.

#### Altri dettagli:

- **media_loc_mean**:
    La media dei valori ottenuti applicando un filtro di media locale (finestra 3x3) sulla matrice delle prese.
    In pratica, per ogni posizione della matrice, si calcola la media dei valori nelle celle vicine (in una finestra 3x3 centrata su quella cella), poi si fa la media di tutti questi valori. Un valore alto di media_loc_mean indica che, in media, le prese sono raggruppate (molte finestre 3x3 hanno almeno una presa).

- **media_loc_std**:
    La deviazione standard dei valori ottenuti dal filtro di media locale (finestra 3x3).
    Indica quanto variano le densità locali di prese sulla board. Un valore alto di media_loc_std indica che la densità locale delle prese è molto variabile (ci sono zone molto dense e zone vuote).

- **centroide_y**:
    Coordinata y (verticale) del baricentro delle prese usate. Valori bassi = prese in basso, valori alti = prese in alto.

- **centroide_x**:
    Coordinata x (orizzontale) del baricentro delle prese usate. Valori bassi = prese a sinistra, valori alti = prese a destra.

- **sim_vert**:
    Simmetria verticale della disposizione delle prese, calcolata confrontando la parte sinistra e destra della board. Valori vicini a 1 indicano una disposizione molto simmetrica rispetto all’asse verticale centrale.

- **sim_oriz**:
    Simmetria orizzontale della disposizione delle prese, calcolata confrontando la parte alta e bassa della board. Valori vicini a 1 indicano una disposizione molto simmetrica rispetto all’asse orizzontale centrale.

--------------------------------

- **skew_y**:  
  Skewness (asimmetria) della distribuzione delle prese sulle righe, valori positivi indicano che le prese sono concentrate nella parte alta della board, valori negativi nella parte bassa, valori vicini a zero indicano una distribuzione bilanciata tra alto e basso.

- **skew_x**:  
  Skewness (asimmetria) della distribuzione delle prese sulle colonne, valori positivi indicano che le prese sono concentrate a destra, negativi a sinistra, vicino a zero distribuzione bilanciata tra destra e sinistra.

- **entropy**:  
  Entropia della distribuzione delle prese sulle colonne, valori alti indicano che le prese sono distribuite su molte colonne (distribuzione uniforme), valori bassi indicano che sono concentrate su poche colonne.

- **n_clusters**:  
  Numero di cluster di prese connesse (gruppi di prese adiacenti, anche in diagonale), un numero alto indica prese sparse in piccoli gruppi, un numero basso indica prese raggruppate in pochi cluster grandi.

- **mean_cluster_size**:  
  Dimensione media dei cluster (numero medio di prese per cluster), valori alti indicano cluster grandi, valori bassi cluster piccoli e sparsi.

- **max_cluster_size**:  
  Dimensione massima di un cluster (il gruppo più grande di prese connesse), indica quanto può essere grande la zona più densa di prese.

- **clusters_per_hold**:  
  Rapporto tra numero di cluster e numero totale di prese, valori vicini a 1 indicano prese molto sparse, valori vicini a 0 prese molto raggruppate.

- **max_density_3x3**:  
  Massima densità di prese trovata in una finestra 3x3 (il massimo numero di prese in qualsiasi blocco 3x3 della board), valori alti indicano la presenza di zone molto dense, valori bassi prese più disperse.

- **bbox_h**:  
  Altezza della bounding box che contiene tutte le prese (numero di righe coperte), valori alti indicano prese distribuite su molte righe, valori bassi prese concentrate verticalmente.

- **bbox_w**:  
  Larghezza della bounding box che contiene tutte le prese (numero di colonne coperte), valori alti indicano prese distribuite su molte colonne, valori bassi prese concentrate orizzontalmente.

- **aspect_ratio**:  
  Rapporto larghezza/altezza della bounding box (bbox_w / bbox_h), valori >1 indicano prese più distribuite in orizzontale, <1 più in verticale, ≈1 distribuzione quadrata.

- **switch_righe**:  
  Numero totale di transizioni (on/off) tra prese su tutte le righe (cambiamenti tra 0 e 1 da una colonna all’altra), valori alti indicano prese alternate/spaziate sulle righe, valori bassi prese raggruppate senza molte interruzioni.

- **switch_colonne**:  
  Numero totale di transizioni (on/off) tra prese su tutte le colonne (cambiamenti tra 0 e 1 da una riga all’altra), valori alti indicano prese alternate/spaziate sulle colonne, valori bassi prese raggruppate senza molte interruzioni.

## Applicazione al dataset:

In [11]:
# Ordina le colonne da 'A1' a 'K18' (tutte le prese della Moonboard 18x11)
hold_cols = [f"{col}{row}" for col in "ABCDEFGHIJK" for row in range(1, 19)]

# Estrai solo queste colonne e converti in array numpy (n_tracciati, 18, 11)
holds_np = holds[hold_cols].to_numpy().reshape(-1, 18, 11)

# Ora holds_np è pronto per l'estrazione delle feature
print(holds_np.shape)  # (numero_tracciati, 18, 11)

(59506, 18, 11)


In [12]:
# holds_np è già di shape (n_tracciati, 18, 11)
X, feature_names = zip(*[estrai_feature_complete_con_nomi(m) for m in holds_np])
X = np.array(X)
holds_features = pd.DataFrame(X, columns=feature_names[0])
holds_features.head()

,totale,centroide_y,centroide_x,var_y,var_x,sim_vert,sim_oriz,diag_1,diag_2,media_loc_u_3_mean,...,n_clusters,mean_cluster_size,max_cluster_size,clusters_per_hold,max_density_3x3,bbox_h,bbox_w,aspect_ratio,switch_righe,switch_colonne
0,7.0,6.000000,3.000000,3.428571,5.142857,0.933333,0.929293,1.0,2.0,0.035354,...,6.0,1.166667,2.0,0.857143,3.0,7.0,8.0,1.142857,13.0,12.0
1,7.0,11.285714,4.714286,13.632653,12.489796,0.922222,0.929293,1.0,0.0,0.035354,...,7.0,1.000000,1.0,1.000000,2.0,11.0,11.0,1.000000,12.0,13.0
2,7.0,9.428571,6.285714,13.102041,13.061224,0.922222,0.929293,1.0,0.0,0.035354,...,7.0,1.000000,1.0,1.000000,2.0,12.0,10.0,0.833333,12.0,14.0
3,8.0,8.625000,4.000000,15.734375,11.000000,0.944444,0.919192,0.0,0.0,0.040404,...,8.0,1.000000,1.0,1.000000,2.0,14.0,11.0,0.785714,14.0,16.0
4,8.0,6.625000,5.250000,15.734375,7.687500,0.933333,0.919192,1.0,3.0,0.040404,...,7.0,1.142857,2.0,0.875000,2.0,15.0,10.0,0.666667,15.0,13.0


## Merging finale:

In [13]:
# Unisci il DataFrame delle feature con quello originale (merge per indice)
df_merged = pd.concat([df.reset_index(drop=True), holds_features.reset_index(drop=True)], axis=1)

# Visualizza le prime righe
df_merged.head()

,name,grade,userGrade,setby,method,userRating,repeats,isBenchmark,isMaster,upgraded,...,n_clusters,mean_cluster_size,max_cluster_size,clusters_per_hold,max_density_3x3,bbox_h,bbox_w,aspect_ratio,switch_righe,switch_colonne
0,Far from the Madding Crowd,6B+,6B+,other,Feet follow hands,4,24993,True,False,False,...,6.0,1.166667,2.0,0.857143,3.0,7.0,8.0,1.142857,13.0,12.0
1,Wuthering Heights,6B+,6B+,other,Feet follow hands,4,35673,True,False,False,...,7.0,1.000000,1.0,1.000000,2.0,11.0,11.0,1.000000,12.0,13.0
2,Problem 3,6B+,6B+,other,Feet follow hands,4,757,False,False,False,...,7.0,1.000000,1.0,1.000000,2.0,12.0,10.0,0.833333,12.0,14.0
3,HARD TIMES,7A,7A,other,Feet follow hands,5,8670,True,False,False,...,8.0,1.000000,1.0,1.000000,2.0,14.0,11.0,0.785714,14.0,16.0
4,Problem 5,7A,7A,other,Feet follow hands,4,255,False,False,False,...,7.0,1.142857,2.0,0.875000,2.0,15.0,10.0,0.666667,15.0,13.0


In [14]:
cols = list(df_merged.columns)
# Trova le posizioni
grade_idx = cols.index('grade')
grade_v_idx = cols.index('grade_v')
# Rimuovi grade_v dalla posizione attuale
cols.pop(grade_v_idx)
# Inserisci grade_v subito dopo grade
cols.insert(grade_idx + 1, 'grade_v')
# Riordina il DataFrame
df_merged = df_merged[cols]
df_merged.head()

,name,grade,grade_v,userGrade,setby,method,userRating,repeats,isBenchmark,isMaster,...,n_clusters,mean_cluster_size,max_cluster_size,clusters_per_hold,max_density_3x3,bbox_h,bbox_w,aspect_ratio,switch_righe,switch_colonne
0,Far from the Madding Crowd,6B+,V4,6B+,other,Feet follow hands,4,24993,True,False,...,6.0,1.166667,2.0,0.857143,3.0,7.0,8.0,1.142857,13.0,12.0
1,Wuthering Heights,6B+,V4,6B+,other,Feet follow hands,4,35673,True,False,...,7.0,1.000000,1.0,1.000000,2.0,11.0,11.0,1.000000,12.0,13.0
2,Problem 3,6B+,V4,6B+,other,Feet follow hands,4,757,False,False,...,7.0,1.000000,1.0,1.000000,2.0,12.0,10.0,0.833333,12.0,14.0
3,HARD TIMES,7A,V6,7A,other,Feet follow hands,5,8670,True,False,...,8.0,1.000000,1.0,1.000000,2.0,14.0,11.0,0.785714,14.0,16.0
4,Problem 5,7A,V6,7A,other,Feet follow hands,4,255,False,False,...,7.0,1.142857,2.0,0.875000,2.0,15.0,10.0,0.666667,15.0,13.0


In [15]:
df_merged.shape

(59506, 448)

In [16]:
output_path = 'C:/Users/Francesco/Documents/uni/tesi/data_expl_analysis/moonboard_2016_engeneered.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8')

print(f"Dataset pulito salvato in: {output_path}")

Dataset pulito salvato in: C:/Users/Francesco/Documents/uni/tesi/data_expl_analysis/moonboard_2016_engeneered.csv
